# 车辆让行预测——lora微调

### Step1导入相关包

In [ ]:
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,Trainer
from datasets import Dataset
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
import json
from transformers import TrainingArguments
from peft import prepare_model_for_kbit_training
from peft import PeftModel
import torch
from typing import Dict
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

### step2 加载数据集合

In [ ]:
with open('../dataset/train_dataset.json', 'r', encoding='utf-8') as f: 
    data = json.load(f)


print(data[0])

# 转换为Dataset格式
dataset = Dataset.from_list(data)
print(dataset)


### step3 加载模型

In [ ]:
#4-bit量化配置（节省显存）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# bnb_config = BitsAndBytesConfig(
#     load_in_8bit=True,  # 改为 8-bit
#     llm_int8_threshold=6.0,  # 8-bit 量化阈值
#     llm_int8_has_fp16_weight=False,  # 不使用 fp16 权重
# )

In [ ]:

model_path="../model/Qwen3-4B-CRAFT-stage1"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)


### Step4 添加分词器

In [ ]:
# 加载模型和tokenizer
token_model_path="../model/Qwen3-4B"
tokenizer = AutoTokenizer.from_pretrained(
    token_model_path,
    padding_side="left",
    trust_remote_code=True
)
print(tokenizer)

### Step5 数据预处理

##### 知识注入

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# 加载 embedding 模型
Embeddingmodel = SentenceTransformer('../model/Qwen3-Embedding-0.6B')

# 初始化 Chroma 客户端
chroma_client = chromadb.PersistentClient(path="../dataset/driver_yield_vector_db")

collection = chroma_client.get_collection(name="driver_yield_knowledge-28")


def retrieve_knowledge(query_text, top_k=8):
    """
    根据 input 文本检索最相关的领域知识
    """
    query_embedding = Embeddingmodel.encode(query_text).tolist()

    results = collection.query(query_embeddings=[query_embedding],n_results=top_k)

    retrieved_docs = results["documents"][0]  # list[str]

    # 拼接成一个字符串
    knowledge_text = "\n".join(retrieved_docs)

    return knowledge_text



In [ ]:
def process_func(example):
    MAX_LENGTH = 4096
    input_ids, attention_mask, labels = [], [], []
    
    retrieved_knowledge = retrieve_knowledge(example['input'], top_k=8)

    enhanced_instruction = (
        example["instruction"]
        + "\n\n### Relevant Domain Knowledge:\n"
        + retrieved_knowledge
    )

    instruction = tokenizer(
        f"<|im_start|>system\n{enhanced_instruction}<|im_end|>"
        + f"\n<|im_start|>user\n{example['input']}<|im_end|>\n"
        + "<|im_start|>assistant\n",
        add_special_tokens=False,
    )

    response = tokenizer(
        example['output'],
        add_special_tokens=False
    )

    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]

    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


tokenized_dataset = dataset.map(process_func, remove_columns=dataset.column_names)

tokenized_dataset

In [ ]:
tokenizer.decode(tokenized_dataset[1]["input_ids"])
# tokenized_dataset[1]["input_ids"]

In [ ]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_dataset[1]["labels"])))

In [ ]:
print(tokenized_dataset["labels"][0])

### Step6划分训练和测试集

In [ ]:
split_dataset = tokenized_dataset.train_test_split(
    test_size=0.1,  # 测试集比例，按需调整
    seed=42,  # 固定随机种子，保证实验可复现
    shuffle=True
)

# 提取划分后的训练集和测试集
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]
print(train_dataset)
print(test_dataset)

### step 7设置训练参数

lora参数

In [ ]:
lora_config = LoraConfig(
    r=8,  # LoRA秩（越大表达能力越强，显存消耗越高）
    lora_alpha=16,  # 缩放因子
    target_modules=[  # QWen3的注意力层模块名（需根据模型结构调整）
        "q_proj", "k_proj", "v_proj", "o_proj",  # 注意力Q/K/V/O投影层
        "gate_proj", "up_proj", "down_proj"  # FFN层（可选，增强微调效果）
    ],
    lora_dropout=0.05,
    fan_in_fan_out=False,
    bias="none",  # 不微调偏置参数
    task_type="CAUSAL_LM"  # 因果语言模型任务
)

# 将LoRA应用到模型
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
for name, parameter in model.named_parameters():
    print(name," 模型的激活状态 ",parameter.requires_grad)

**加载评估函数**

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

def compute_metrics(eval_preds):

    preds, labels = eval_preds

    labels = labels[:, 1:]
    preds = preds[:, :-1]

    true_predictions = [
        [p for (p, l) in zip(pred, gold_label) if l != -100]
        for pred, gold_label in zip(preds, labels)
    ]
    true_labels = [
        [l for (p, l) in zip(pred, gold_label) if l != -100]
        for pred, gold_label in zip(preds, labels)
    ]
    
    ID2LABEL = {
        tokenizer.convert_tokens_to_ids("1"): 1,  # yield → 正类
        tokenizer.convert_tokens_to_ids("0"): 0  # no_yield → 负类
    }


    y_pred = np.array([ID2LABEL[p[-1]] for p in true_predictions])
    y_true = np.array([ID2LABEL[l[-1]] for l in true_labels])


    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred)
    }

    return metrics


def preprocess_logits_for_metrics(logits, labels): 
    if isinstance(logits, tuple): 
        # Depending on the model and config, logits may contain extra tensors, 
        # # like past_key_values, but logits always come first 
        logits = logits[0] 

    return logits.argmax(dim=-1)



### 加载训练参数


In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoint/Qwen3-4B-CRAFT",  # 模型保存路径
    per_device_train_batch_size=2,  
    gradient_accumulation_steps=8,  
    learning_rate=2e-5,  
    num_train_epochs=3, 
    logging_steps=10,
    save_steps=50,
    bf16=True,

    optim="paged_adamw_8bit",  
    report_to="none",  
    eval_strategy="steps",
    eval_steps=10,  
    per_device_eval_batch_size=2,  

    load_best_model_at_end=True, 
)

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    data_collator=transformers.DataCollatorForSeq2Seq(
        tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
    
    args=training_args,
)


### Step 9 开始训练

In [ ]:
print(model.hf_device_map)
trainer.train()